In [4]:
# ── Cell 1: SLE-guided negative downsampling → balanced 1:1 train set ────────
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# ── Project config ────────────────────────────────────────────────────────────
sys.path.append(str(Path("..").resolve()))
from src.config import ROOT_DIR

ROOT_DIR  = Path(ROOT_DIR)
ML_DIR    = ROOT_DIR / "ML"
SCO_DIR   = ML_DIR  / "MIC_StenosisDetection_OutputFolder_SEL2"
OUT_DIR   = ML_DIR  / "downsampled_SEL2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NEG_SCO   = SCO_DIR / "10-fold-neg.SEL(002).sco"
POS_SCO   = SCO_DIR / "10-fold-pos.SEL(002).sco"
NEG_CSV   = ML_DIR  / "neg.csv"
POS_CSV   = ML_DIR  / "pos.csv"

# ── Load SLE scores ───────────────────────────────────────────────────────────
def load_sco(path):
    return pd.read_csv(path, sep=r'\s+', skiprows=1,
                       header=None, names=['sample_idx', 'score'])

neg_scores = load_sco(NEG_SCO)
pos_scores = load_sco(POS_SCO)

print(f"Negatives scored : {len(neg_scores):>6}")
print(f"Positives scored : {len(pos_scores):>6}")

# ── Select the N most difficult negatives (highest SLE score) ─────────────────
N = len(pos_scores)
hard_neg_idx = (neg_scores
                .sort_values('score', ascending=False)
                .head(N)
                ['sample_idx']
                .values)

threshold = neg_scores.set_index('sample_idx').loc[hard_neg_idx, 'score'].min()
print(f"Keeping top-{N} hard negatives (score cutoff: {threshold:.4f})")

# ── Load feature CSVs (no header — raw feature rows) ─────────────────────────
neg_all = pd.read_csv(NEG_CSV, header=None)
pos_all = pd.read_csv(POS_CSV, header=None)

if len(neg_all) != len(neg_scores):
    print(f"⚠️  Warning: neg.csv has {len(neg_all)} rows but .sco has {len(neg_scores)} — difference of {len(neg_all)-len(neg_scores)}")
    print(f"   Trimming neg.csv to match .sco length")
    neg_all = neg_all.iloc[:len(neg_scores)]

if len(pos_all) != len(pos_scores):
    print(f"⚠️  Warning: pos.csv has {len(pos_all)} rows but .sco has {len(pos_scores)} — difference of {len(pos_all)-len(pos_scores)}")
    print(f"   Trimming pos.csv to match .sco length")
    pos_all = pos_all.iloc[:len(pos_scores)]

# ── Build balanced dataset ────────────────────────────────────────────────────
neg_hard = neg_all.iloc[hard_neg_idx].copy()
neg_hard['label'] = 0
pos_all  = pos_all.copy()
pos_all['label'] = 1

balanced = (pd.concat([pos_all, neg_hard], ignore_index=True)
              .sample(frac=1, random_state=42)
              .reset_index(drop=True))

print(f"\nBalanced train set : {len(balanced)} samples "
      f"({(balanced['label']==1).sum()} pos / {(balanced['label']==0).sum()} neg)")

# ── Save ──────────────────────────────────────────────────────────────────────
balanced.to_csv(OUT_DIR / "train_balanced_SEL2.csv", index=False, header=False)
print(f"Saved → {OUT_DIR / 'train_balanced_SEL2.csv'}")

Negatives scored :  29367
Positives scored :   2108
Keeping top-2108 hard negatives (score cutoff: 0.7925)

Balanced train set : 4216 samples (2108 pos / 2108 neg)
Saved → C:\Users\ferra\MIC\1r_any_UNICAS\2n_Semestre\Image_Processing_and_Analysis\project\MIC_project\Proposal_StenosisDetection\ML\downsampled_SEL2\train_balanced_SEL2.csv
